In [6]:
!pip install ipytest

In [7]:
import ipytest
import pytest
ipytest.autoconfig()

In [8]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.base import BaseEstimator, TransformerMixin
from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA
from sklearn.pipeline import Pipeline
import matplotlib.pyplot as plt


class MedianOutlierHandler(BaseEstimator, TransformerMixin):
    def fit(self, X, y=None):
        """
        Calcula las medianas y los límites para detección de outliers usando IQR.
        También guarda las medianas para imputación de valores nulos.
        """
        self.medians_ = X.median()
        self.bounds_ = {}
        for col in X.columns:
            Q1 = X[col].quantile(0.25)
            Q3 = X[col].quantile(0.75)
            IQR = Q3 - Q1
            lower = Q1 - 1.5 * IQR
            upper = Q3 + 1.5 * IQR
            self.bounds_[col] = (lower, upper)
        return self

    def transform(self, X):
        """
        Imputa valores nulos con la mediana y reemplaza outliers por la mediana.
        """
        X_copy = X.copy()
        for col in X_copy.columns:
            median = self.medians_[col]
            lower, upper = self.bounds_[col]
            X_copy[col] = X_copy[col].fillna(median)
            X_copy[col] = X_copy[col].apply(lambda x: median if x < lower or x > upper else x)
        return X_copy

class CorrelationRemover(BaseEstimator, TransformerMixin):
    def fit(self, X, y=None):
        corr_matrix = X.corr().abs()
        upper_triangle = corr_matrix.where(np.triu(np.ones(corr_matrix.shape), k=1).astype(bool))
        self.to_drop_ = [column for column in upper_triangle.columns if any(upper_triangle[column] > 0.75)]
        return self

    def transform(self, X):
        return X.drop(columns=self.to_drop_, errors='ignore')

class InsurancePipeline:
    def __init__(self, df: pd.DataFrame):
        self.df = df.copy()
        self.X_train = None
        self.X_test = None
        self.y_train = None
        self.y_test = None
        self.pipeline = None
        self.n_components_90 = None

    def preprocess(self):
        self.df = self.df.drop_duplicates()

        # Filtrado por porcentaje de nulos (<=5%)
        porcentaje_nulos_por_fila = self.df.isnull().mean(axis=1) * 100
        self.df = self.df[porcentaje_nulos_por_fila <= 5]

        # Separar X de y
        X = self.df.iloc[:, :-1]
        y = self.df.iloc[:, -1]

        # División train/test
        self.X_train, self.X_test, self.y_train, self.y_test = train_test_split(X, y, test_size=0.2, random_state=42)

        # Construcción del pipeline
        self.pipeline = Pipeline([
            ('outlier_handler', MedianOutlierHandler()),
            ('corr_remover', CorrelationRemover()),
            ('scaler', StandardScaler()),
            ('pca', PCA())  # PCA inicial para determinar componentes
        ])

        # Ajustar pipeline en X_train
        X_train_transformed = self.pipeline.fit_transform(self.X_train)

        # Determinar número de componentes para explicar >=90% varianza
        cumulative_variance = np.cumsum(self.pipeline.named_steps['pca'].explained_variance_ratio_)
        self.n_components_90 = np.argmax(cumulative_variance >= 0.90) + 1

        # Ajustar PCA con n_components_90
        self.pipeline.named_steps['pca'].n_components = self.n_components_90
        X_train_final = self.pipeline.fit_transform(self.X_train)
        X_test_final = self.pipeline.transform(self.X_test)

        return X_train_final, X_test_final, self.y_train, self.y_test

In [9]:
@pytest.fixture
def sample_df():
    # Genera un DataFrame de prueba con valores numéricos y algunos nulos
    np.random.seed(42)
    data = np.random.randn(100, 5)
    df = pd.DataFrame(data, columns=[f'feature_{i}' for i in range(5)])
    df.iloc[0, 0] = np.nan  # Introducir un nulo
    df['target'] = np.random.choice([0, 1], size=100)
    return df

In [10]:
%%ipytest

'''Verifica que los nulos y outliers se imputan correctamente con la mediana.'''
def test_median_outlier_handler(sample_df):
    X = sample_df.drop(columns=['target'])
    handler = MedianOutlierHandler()
    handler.fit(X)
    X_transformed = handler.transform(X)

    # No debe haber nulos
    assert X_transformed.isnull().sum().sum() == 0

    # Verifica que los valores estén dentro de los límites
    for col in X.columns:
        lower, upper = handler.bounds_[col]
        assert X_transformed[col].between(lower, upper).all()

'''Verifica que se eliminen columnas altamente correlacionadas'''
def test_correlation_remover(sample_df):
    X = sample_df.drop(columns=['target'])
    # Introducir correlación artificial
    X['feature_4'] = X['feature_0'] * 0.95
    remover = CorrelationRemover()
    remover.fit(X)
    X_transformed = remover.transform(X)

    # Verifica que se haya eliminado al menos una columna
    assert 'feature_4' not in X_transformed.columns

'''Verifica que el pipeline se construya y transforme los datos correctamente'''
def test_pipeline_preprocess(sample_df):
    pipeline = InsurancePipeline(sample_df)
    X_train_final, X_test_final, y_train, y_test = pipeline.preprocess()

    # Verifica que las salidas tengan la forma esperada
    assert X_train_final.shape[0] == len(y_train)
    assert X_test_final.shape[0] == len(y_test)

    # Verifica que se haya calculado el número de componentes
    assert pipeline.n_components_90 is not None
    assert pipeline.n_components_90 > 0


...                                                                                          [100%]
3 passed in 0.10s
